In [16]:
# Cell 1 — Imports, paths & helpers
import os, re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib

warnings.filterwarnings("ignore")

# --- Paths ---
DATA_MERGED   = Path("./data_thuydien/merged_hoa_binh_keyjoin.csv")  # TRAIN
DATA_ENRICHED = Path("./data_thuydien/data_thuydien_enriched.csv")   # DEMO/INFER

OUT_DIR     = Path("./checkpoint/dtree")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH  = OUT_DIR / "dtree_model.pkl"      # lưu cả pipeline (imputer + tree)
FEATS_PATH  = OUT_DIR / "features_used.json"
META_PATH   = OUT_DIR / "train_meta.json"
PRED_PATH   = OUT_DIR / "predictions_test.csv"
FIG_TS_PATH = OUT_DIR / "plot_test_timeseries.png"
FIG_PP_PATH = OUT_DIR / "plot_test_parity.png"
FIG_FI_PATH = OUT_DIR / "plot_feature_importance.png"

# --- Helpers ---
def normalize_col(c: str) -> str:
    c0 = re.sub(r"\s+", " ", str(c).strip().lower())
    c0 = re.sub(r"\s*\([^)]*\)", "", c0)  # bỏ đơn vị trong ngoặc
    c0 = c0.replace("%", "pct").replace("°", "")
    c0 = re.sub(r"[^a-z0-9_ ]+", "_", c0).replace(" ", "_")
    c0 = re.sub(r"_+", "_", c0).strip("_")
    return c0

def find_first_col(cols, *keywords):
    for c in cols:
        low = c.lower()
        if all(k in low for k in keywords):
            return c
    return None

def parse_best_datetime(series):
    s1 = pd.to_datetime(series, errors="coerce", dayfirst=False)
    s2 = pd.to_datetime(series, errors="coerce", dayfirst=True)
    return s1 if s1.notna().sum() >= s2.notna().sum() else s2

def to_numeric_safe(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.replace(",", ".", regex=False).str.replace(" ", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

print("✓ Helpers ready.")

✓ Helpers ready.


In [17]:
# Cell 2 — Load (TRAIN) merged & detect columns
assert DATA_MERGED.exists(), f"Không tìm thấy file TRAIN: {DATA_MERGED}"
raw = pd.read_csv(DATA_MERGED)

norm_map = {c: normalize_col(c) for c in raw.columns}
df = raw.rename(columns=norm_map).copy()

time_col = "thoi_diem" if "thoi_diem" in df.columns else ("time" if "time" in df.columns else None)
assert time_col is not None, f"Thiếu cột thời gian trong merged: {df.columns.tolist()}"

col_target = find_first_col(df.columns, "muc","nuoc","thuong","luu")
col_flow   = "luu_luong_den_ho_m3_s_capped" if "luu_luong_den_ho_m3_s_capped" in df.columns else (
             "luu_luong_den_ho_m3_s" if "luu_luong_den_ho_m3_s" in df.columns else
             find_first_col(df.columns, "luu","luong","den","ho"))

col_temp = "temp_c"    if "temp_c"    in df.columns else find_first_col(df.columns, "temp","temperature")
col_rh   = "rh_pct"    if "rh_pct"    in df.columns else find_first_col(df.columns, "rh","humidity")
col_pr   = "precip_mm" if "precip_mm" in df.columns else find_first_col(df.columns, "precip","mua")
col_cc   = "cloud_pct" if "cloud_pct" in df.columns else find_first_col(df.columns, "cloud")

print("TRAIN columns (merged):")
print(" - time_col:", time_col)
print(" - target  :", col_target)
print(" - inflow  :", col_flow)
print(" - weather :", [col_temp, col_rh, col_pr, col_cc])

# Parse & sort
dt = parse_best_datetime(df[time_col])
df = df.assign(thoi_diem=dt).dropna(subset=["thoi_diem"]).sort_values("thoi_diem").reset_index(drop=True)
if time_col != "thoi_diem":
    df = df.drop(columns=[time_col])

keep = ["thoi_diem"] + [c for c in [col_target, col_flow, col_temp, col_rh, col_pr, col_cc] if c is not None]
df = df[keep].copy()

for c in keep:
    if c == "thoi_diem": continue
    df[c] = to_numeric_safe(df[c])

print("[TRAIN] Shape after select:", df.shape)
df.head(3)


TRAIN columns (merged):
 - time_col: thoi_diem
 - target  : muc_nuoc_thuong_luu_m
 - inflow  : luu_luong_den_ho_m3_s_capped
 - weather : ['temp_c', 'rh_pct', 'precip_mm', 'cloud_pct']
[TRAIN] Shape after select: (33264, 7)


,thoi_diem,muc_nuoc_thuong_luu_m,luu_luong_den_ho_m3_s_capped,temp_c,rh_pct,precip_mm,cloud_pct
0,2022-01-01 00:00:00,112.13,848.0,14.5,94,0.0,65
1,2022-01-01 01:00:00,112.14,700.0,14.6,94,0.0,100
2,2022-01-01 02:00:00,112.14,700.0,14.9,92,0.0,100


In [18]:
# Cell 3 — Feature engineering (TRAIN, strictly-past 28 ngày)
WEATHER_COLS = [c for c in [col_temp, col_rh, col_pr, col_cc] if c is not None]
VAR_LIST = [c for c in [col_target, col_flow] + WEATHER_COLS if c is not None]

df = df.set_index("thoi_diem").sort_index()

feat = pd.DataFrame(index=df.index)
feat["hour"] = feat.index.hour
feat["dow"]  = feat.index.dayofweek
feat["month"]= feat.index.month
feat["is_weekend"] = (feat["dow"] >= 5).astype(int)

def add_roll_feats(dst: pd.DataFrame, src: pd.Series, name: str, windows=("24H","3D","7D","14D","28D")):
    s = src.shift(1)  # strictly-past
    for w in windows:
        r = s.rolling(w, closed="left")
        dst[f"{name}__mean_{w.lower()}"] = r.mean()
        dst[f"{name}__std_{w.lower()}"]  = r.std()
        dst[f"{name}__max_{w.lower()}"]  = r.max()
        dst[f"{name}__min_{w.lower()}"]  = r.min()

def add_lag_feats(dst: pd.DataFrame, src: pd.Series, name: str, lags=(1,3,6,12,24,48,72)):
    for k in lags:
        dst[f"{name}__lag_{k}"] = src.shift(k)

for c in VAR_LIST:
    add_roll_feats(feat, df[c], c)
    add_lag_feats(feat, df[c], c)

y = df[col_target].copy()
data = pd.concat([feat, y.rename("target")], axis=1).dropna(subset=["target"])

print("Feature matrix:", feat.shape, "| Data rows:", data.shape[0])
print("Example columns:", list(feat.columns)[:10])

Feature matrix: (33264, 166) | Data rows: 33264
Example columns: ['hour', 'dow', 'month', 'is_weekend', 'muc_nuoc_thuong_luu_m__mean_24h', 'muc_nuoc_thuong_luu_m__std_24h', 'muc_nuoc_thuong_luu_m__max_24h', 'muc_nuoc_thuong_luu_m__min_24h', 'muc_nuoc_thuong_luu_m__mean_3d', 'muc_nuoc_thuong_luu_m__std_3d']


In [19]:
# Cell 4 — Split 80/20 theo thời gian
N = len(data)
split_idx = int(N * 0.8)
train = data.iloc[:split_idx].copy()
test  = data.iloc[split_idx:].copy()

X_train, y_train = train.drop(columns=["target"]), train["target"]
X_test,  y_test  = test.drop(columns=["target"]),  test["target"]

print(f"Tổng số mẫu: {N:,} -> Train: {len(train):,} | Test: {len(test):,}")
print(f"Số feature: {X_train.shape[1]}")
print("Khoảng thời gian train:", X_train.index.min(), "->", X_train.index.max())
print("Khoảng thời gian test :", X_test.index.min(),  "->", X_test.index.max())

Tổng số mẫu: 33,264 -> Train: 26,611 | Test: 6,653
Số feature: 166
Khoảng thời gian train: 2022-01-01 00:00:00 -> 2025-01-13 18:00:00
Khoảng thời gian test : 2025-01-13 19:00:00 -> 2025-10-17 23:00:00


In [20]:
# Cell 5 — Train Decision Tree (Pipeline: SimpleImputer + Tree)
# DecisionTreeRegressor KHÔNG nhận NaN -> dùng SimpleImputer(strategy='median')
model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("reg", DecisionTreeRegressor(
        random_state=42,
        max_depth=12,           # bạn có thể tinh chỉnh
        min_samples_leaf=20,    # giúp tổng quát hơn, bớt overfit
        min_samples_split=40
    ))
])

model.fit(X_train, y_train)
print("✓ Decision Tree trained.")

✓ Decision Tree trained.


In [21]:
# Cell 6 — Evaluate & save artifacts (pipeline)
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return (np.abs((y_true[mask]-y_pred[mask]) / y_true[mask])).mean() * 100

yhat_tr = model.predict(X_train)
yhat_te = model.predict(X_test)

metrics = {
    "train": {"MAE": float(mean_absolute_error(y_train, yhat_tr)),
              "RMSE": float(mean_squared_error(y_train, yhat_tr, squared=False)),
              "R2": float(r2_score(y_train, yhat_tr)),
              "MAPE_pct": float(mape(y_train, yhat_tr))},
    "test":  {"MAE": float(mean_absolute_error(y_test, yhat_te)),
              "RMSE": float(mean_squared_error(y_test, yhat_te, squared=False)),
              "R2": float(r2_score(y_test, yhat_te)),
              "MAPE_pct": float(mape(y_test, yhat_te))},
    "model_type": "DecisionTreeRegressor + SimpleImputer(median)"
}

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Save pipeline + features + predictions
joblib.dump(model, MODEL_PATH)
with open(FEATS_PATH, "w", encoding="utf-8") as f:
    json.dump(X_train.columns.tolist(), f, ensure_ascii=False, indent=2)

pred_df = pd.DataFrame({"thoi_diem": X_test.index, "y_true": y_test.values, "y_pred": yhat_te})
pred_df.to_csv(PRED_PATH, index=False)

print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Saved:")
print(" - MODEL_PATH:", MODEL_PATH)
print(" - FEATS_PATH:", FEATS_PATH)
print(" - META_PATH :", META_PATH)
print(" - PRED_PATH :", PRED_PATH)

{
  "train": {
    "MAE": 0.01987388454816597,
    "RMSE": 0.04373213297984406,
    "R2": 0.9999331777904781,
    "MAPE_pct": 0.01821864768020251
  },
  "test": {
    "MAE": 1.3385835361984226,
    "RMSE": 3.239189646901914,
    "R2": 0.8648792067800503,
    "MAPE_pct": 1.5268813702858108
  },
  "model_type": "DecisionTreeRegressor + SimpleImputer(median)"
}
Saved:
 - MODEL_PATH: checkpoint/dtree/dtree_model.pkl
 - FEATS_PATH: checkpoint/dtree/features_used.json
 - META_PATH : checkpoint/dtree/train_meta.json
 - PRED_PATH : checkpoint/dtree/predictions_test.csv


In [22]:
# Cell 7 — Feature importance (Decision Tree)
fi_df = None
try:
    reg = model.named_steps["reg"]
    importances = getattr(reg, "feature_importances_", None)
    if importances is not None:
        fi_df = pd.DataFrame({"feature": X_train.columns, "importance": importances}).sort_values("importance", ascending=False)
except Exception:
    fi_df = None

if fi_df is not None and not fi_df.empty:
    top_k = min(30, len(fi_df))
    plt.figure(figsize=(10, max(4, int(0.25*top_k)+2)))
    plt.barh(fi_df["feature"].head(top_k)[::-1], fi_df["importance"].head(top_k)[::-1])
    plt.xlabel("Importance"); plt.title("Top feature importance (Decision Tree)")
    plt.tight_layout(); plt.savefig(FIG_FI_PATH, dpi=150); plt.close()
    print("Saved feature importance:", FIG_FI_PATH)
else:
    print("No feature importance available.")

Saved feature importance: checkpoint/dtree/plot_feature_importance.png


In [23]:
# Cell 8 — Plots (time-series + parity)
# Time-series 500 điểm cuối
k = 500
sl = slice(max(0, len(pred_df)-k), len(pred_df))
plt.figure(figsize=(12,4))
plt.plot(pred_df["thoi_diem"].values[sl], pred_df["y_true"].values[sl], label="Thực tế")
plt.plot(pred_df["thoi_diem"].values[sl], pred_df["y_pred"].values[sl], label="Dự báo")
plt.title("Test — y_true vs y_pred (500 điểm cuối)")
plt.xlabel("Thời điểm"); plt.ylabel("Mực nước (m)")
plt.xticks(rotation=25); plt.legend(); plt.tight_layout()
plt.savefig(FIG_TS_PATH, dpi=150); plt.close()

# Parity
plt.figure(figsize=(5,5))
plt.scatter(pred_df["y_true"].values, pred_df["y_pred"].values, s=6, alpha=0.6)
mn = float(min(pred_df["y_true"].min(), pred_df["y_pred"].min()))
mx = float(max(pred_df["y_true"].max(), pred_df["y_pred"].max()))
plt.plot([mn, mx], [mn, mx])
plt.title("Parity plot — Test (Decision Tree)")
plt.xlabel("y_true (m)"); plt.ylabel("y_pred (m)")
plt.tight_layout()
plt.savefig(FIG_PP_PATH, dpi=150); plt.close()

print("Saved figures:")
print(" - TS:", FIG_TS_PATH)
print(" - PP:", FIG_PP_PATH)

Saved figures:
 - TS: checkpoint/dtree/plot_test_timeseries.png
 - PP: checkpoint/dtree/plot_test_parity.png


In [24]:
# --- Chèn vào sau khi df_en đã được parse & rename trong Cell 9 ---

# 0) Tên file features đã dùng lúc train
FEATS_PATH = Path("./checkpoint/dtree/features_used.json")  # hoặc path tới features của XGB (chắc bạn đã đặt)
if not FEATS_PATH.exists():
    # fallback: path theo xgb checkpoint (nếu bạn train XGB trước)
    FEATS_PATH = Path("./checkpoint/xgb/features_used.json")

feat_list = None
if FEATS_PATH.exists():
    import json
    feat_list = json.load(open(FEATS_PATH, "r", encoding="utf-8"))
    print("[INFO] Loaded feat_list from:", FEATS_PATH)
else:
    print("[WARN] Không tìm thấy features_used.json tại", FEATS_PATH, "- sẽ suy tính từ enriched.")

# 1) Tên inflow hiện có trong enriched (raw)
has_raw_inflow = None
for cand in ["luu_luong_den_ho_m3_s", "luu_luong_den_ho", "luu_luong", "inflow"]:
    if cand in df_en.columns:
        has_raw_inflow = cand
        break

print("[DEBUG] enriched has raw inflow column:", has_raw_inflow)

# 2) Tìm tên inflow mong đợi từ feat_list (nếu có)
expected_inflow_name = None
if feat_list is not None:
    for f in feat_list:
        lf = f.lower()
        if "luu_luong" in lf or "luu_luong_den" in lf or "inflow" in lf:
            # chọn tên cột chính (chỉ tên cơ sở, trước dấu __ của feature-engineer)
            # Các feature-engineer đang tạo cột như "luu_luong_den_ho_m3_s__lag_1" v.v.
            # Ta lấy phần tiền tố trước "__"
            base = f.split("__")[0]
            expected_inflow_name = base
            break
    print("[DEBUG] expected inflow base from feat_list:", expected_inflow_name)

# 3) Nếu feat_list mong đợi '..._capped' mà enriched chỉ có raw -> tạo cột capped
def create_capped_from_raw(df_source, raw_col, capped_col, cap_value=None):
    """Tạo capped_col trong df_source từ raw_col.
       Nếu cap_value None -> chỉ copy (no capping). Nếu cap_value numeric -> clip upper.
    """
    if raw_col not in df_source.columns:
        df_source[capped_col] = np.nan
        return
    if cap_value is None:
        df_source[capped_col] = df_source[raw_col].copy()
    else:
        df_source[capped_col] = df_source[raw_col].clip(upper=cap_value)

# Decide action
if expected_inflow_name is not None and ("capped" in expected_inflow_name):
    # expected '..._capped' but enriched may not have it
    if expected_inflow_name not in df_en.columns:
        if has_raw_inflow is None:
            # enriched thiếu raw inflow too -> create NaN column; inference will still run (imputer)
            df_en[expected_inflow_name] = np.nan
            print(f"[WARN] enriched không có inflow -> tạo cột {expected_inflow_name} = NaN.")
        else:
            # Option A (simple, quick): copy raw -> same name expected_inflow_name
            df_en[expected_inflow_name] = df_en[has_raw_inflow].copy()
            print(f"[INFO] Tạo cột {expected_inflow_name} từ {has_raw_inflow} bằng copy (no capping).")

            # Option B (recommended if bạn muốn consistency với train):
            # Tính cap_value từ tập TRAIN (merged) theo percentile (ví dụ 99.5%) và áp clipping.
            # Uncomment phần dưới nếu bạn muốn áp cap thống nhất với train.
            #
            # TRAIN_MERGED = Path("./data_thuydien/merged_hoa_binh_keyjoin.csv")
            # if TRAIN_MERGED.exists():
            #     tr = pd.read_csv(TRAIN_MERGED)
            #     trcols = [c.lower() for c in tr.columns]
            #     # tìm cột luu_luong trong train (gốc)
            #     cand = next((c for c in tr.columns if "luu_luong" in c.lower()), None)
            #     if cand is not None:
            #         cap_val = tr[cand].quantile(0.995)  # percentile tùy chỉnh
            #         df_en[expected_inflow_name] = df_en[has_raw_inflow].clip(upper=cap_val)
            #         print(f"[INFO] Tạo {expected_inflow_name} bằng clipping raw tại cap={cap_val:.3f} (từ train).")
            #     else:
            #         print("[WARN] Không tìm thấy cột luu_luong trong TRAIN để tính cap; giữ copy.")
            # else:
            #     print("[WARN] Không tìm thấy TRAIN file để tính cap; giữ copy.")
else:
    # expected inflow is None or does not contain 'capped'
    # If feat_list expects plain 'luu_luong...' but enriched has only raw with different naming, align:
    if expected_inflow_name is not None and expected_inflow_name not in df_en.columns:
        # try to create expected column by copying raw inflow if available
        if has_raw_inflow is not None:
            df_en[expected_inflow_name] = df_en[has_raw_inflow].copy()
            print(f"[INFO] Created expected inflow column '{expected_inflow_name}' by copying '{has_raw_inflow}'.")
        else:
            # no inflow in enriched at all -> create NaN col
            df_en[expected_inflow_name] = np.nan
            print(f"[WARN] Không tìm thấy inflow in enriched; tạo '{expected_inflow_name}' = NaN.")
    else:
        # if feat_list not available, ensure minimal inflow col exists for inference functions
        if has_raw_inflow is None:
            print("[WARN] Enriched không có bất kỳ cột inflow nào; inference vẫn chạy nhưng nhiều feature NaN.")
        else:
            # nothing to do, enriched already has raw inflow
            pass

# 4) Bảo đảm df_en có tất cả cột VAR_LIST_INF (nếu VAR_LIST_INF tồn tại)
try:
    VAR_LIST_INF  # nếu đã tồn tại
except NameError:
    # nếu chưa có, dựng biến list từ df_en hiện tại
    VAR_LIST_INF = [c for c in df_en.columns if c != "thoi_diem"]

for c in VAR_LIST_INF:
    if c not in df_en.columns:
        df_en[c] = np.nan
        print(f"[WARN] Tạo cột '{c}' trống trong enriched để inference xử lý.")

# 5) cuối cùng set index & sort
df_en = df_en.reset_index().set_index("thoi_diem").sort_index()
print("[FINAL] enriched columns used:", list(df_en.columns))

[INFO] Loaded feat_list from: checkpoint/dtree/features_used.json
[DEBUG] enriched has raw inflow column: luu_luong_den_ho_m3_s
[DEBUG] expected inflow base from feat_list: luu_luong_den_ho_m3_s_capped
[FINAL] enriched columns used: ['index', 'muc_nuoc_thuong_luu_m', 'luu_luong_den_ho_m3_s', 'temp_c', 'rh_pct', 'precip_mm', 'cloud_pct', 'luu_luong_den_ho_m3_s_capped']


In [25]:
# Cell 10 — Inference: anchor vào bản ghi gần nhất trước t (Decision Tree + Imputer)
TARGET_TIME = "2025-10-25 07:00"  # ví dụ t khá xa, enriched chỉ có tới 20 -> anchor = 20

# Load pipeline & feature list
pipe = joblib.load(MODEL_PATH)   # SimpleImputer + DecisionTree
with open(FEATS_PATH, "r", encoding="utf-8") as f:
    feat_list = json.load(f)

def build_features_at_time_from_enriched(frame: pd.DataFrame, t: pd.Timestamp, vars_used: list,
                                         min_days_coverage: int = 0) -> pd.DataFrame:
    # 1) Anchor
    t = pd.to_datetime(t)
    idx = frame.index[frame.index < t]
    if len(idx) == 0:
        raise ValueError("Enriched không có quan sát nào trước thời điểm yêu cầu.")
    t_anchor = idx.max()

    # 2) Strictly-past tới anchor (loại chính anchor)
    sub = frame.loc[:t_anchor].iloc[:-1]

    # 3) (tuỳ chọn) cảnh báo độ phủ trong 28D
    if min_days_coverage > 0:
        left_edge = t_anchor - pd.Timedelta(days=28)
        coverage_cnt = sub.loc[left_edge:t_anchor].shape[0]
        if coverage_cnt < min_days_coverage:
            print(f"[WARN] Vùng 28D trước anchor có {coverage_cnt} bản ghi (<{min_days_coverage}). Vẫn dự báo (imputer sẽ xử lý).")

    out = {}
    # 4) Time features từ chính t
    out["hour"] = t.hour
    out["dow"] = t.dayofweek
    out["month"] = t.month
    out["is_weekend"] = int(out["dow"] >= 5)

    # 5) Helpers
    def roll_stats(s, prefix):
        r24  = s.rolling("24H", closed="left").agg(["mean","std","max","min"])
        r3d  = s.rolling("3D",  closed="left").agg(["mean","std","max","min"])
        r7d  = s.rolling("7D",  closed="left").agg(["mean","std","max","min"])
        r14d = s.rolling("14D", closed="left").agg(["mean","std","max","min"])
        r28d = s.rolling("28D", closed="left").agg(["mean","std","max","min"])
        last = {}
        for name, robj in [("24h", r24), ("3d", r3d), ("7d", r7d), ("14d", r14d), ("28d", r28d)]:
            last[f"{prefix}__mean_{name}"] = robj["mean"].iloc[-1] if len(robj) else np.nan
            last[f"{prefix}__std_{name}"]  = robj["std"].iloc[-1]  if len(robj) else np.nan
            last[f"{prefix}__max_{name}"]  = robj["max"].iloc[-1]  if len(robj) else np.nan
            last[f"{prefix}__min_{name}"]  = robj["min"].iloc[-1]  if len(robj) else np.nan
        return last

    def add_lags(s, prefix, lags=(1,3,6,12,24,48,72)):
        for k in lags:
            out[f"{prefix}__lag_{k}"] = s.shift(k).iloc[-1] if len(s) >= k+1 else np.nan

    # 6) Cho từng biến đã dùng khi TRAIN
    for col in vars_used:
        if col not in frame.columns:
            for w in ["24h","3d","7d","14d","28d"]:
                out[f"{col}__mean_{w}"] = np.nan
                out[f"{col}__std_{w}"]  = np.nan
                out[f"{col}__max_{w}"]  = np.nan
                out[f"{col}__min_{w}"]  = np.nan
            for k in [1,3,6,12,24,48,72]:
                out[f"{col}__lag_{k}"] = np.nan
            continue

        s = frame[col].astype(float)
        s.index = pd.to_datetime(frame.index)
        s = s.loc[:t_anchor].iloc[:-1]  # strictly-past
        out.update(roll_stats(s, col))
        add_lags(s, col)

    # 7) Lắp đúng thứ tự cột đã train
    row = pd.DataFrame({k:[v] for k,v in out.items()})
    for c in feat_list:
        if c not in row.columns:
            row[c] = np.nan
    row = row[feat_list]
    return row

# Parse TARGET_TIME linh hoạt
t = pd.to_datetime(TARGET_TIME, dayfirst=True, errors="coerce")
if pd.isna(t):
    t = pd.to_datetime(TARGET_TIME)

# Build row & predict (pipeline tự impute)
rowX = build_features_at_time_from_enriched(df_en, t, VAR_LIST_INF, min_days_coverage=0)
yhat = float(pipe.predict(rowX)[0])
print(f"[Predict @ {t}] (anchor = obs gần nhất trước t)  =>  Mực nước TL ước lượng: {yhat:.4f} m")


[Predict @ 2025-10-25 07:00:00] (anchor = obs gần nhất trước t)  =>  Mực nước TL ước lượng: 115.7092 m
